# Estimating Returns from Validated Cointegrated Spreads

## Imports

In [1]:
import sys
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))

from src.data import get_close_prices, get_market_cap, get_ohlcv
from src.modelling import TICKERS, TRAIN_START, TRAIN_END, TEST_START, TEST_END, INTERVAL, TICKER_NAMES, CANDIDATE_PAIRS
from src.modelling import engle_granger_test, screen_pairs, adf_test # co-integration
from src.modelling import spread_summary, compute_spread, compute_zscore, compute_rolling_half_life, compute_rolling_zscore # spread analysis
from src.modelling import compute_spread_returns, build_spread_return_matrix, historical_mean_return, ewma_return, ou_expected_return, build_ou_expected_returns, sample_covariance, shrinkage_covariance, spread_vs_traditional_estimates

## Load IN-SAMPLE prices (Jan 2019 - Dec 2023) and cointegrated pairs

In [2]:
# Raw prices
is_prices_df = pd.read_csv('../../data/processed/is_prices_df.csv', index_col=0, parse_dates=True)

# Cointegrated pairs
cointegrated_pairs = pd.read_csv("../../data/processed/cointegrated_pairs.csv")

In [3]:
display(is_prices_df)

,NVDA.O,AMD.O,TSM.N,KO.N,PEP.O,JPM.N,BAC.N,GS.N,MS.N,XOM.N,CVX.N,AMZN.O,MSFT.O,META.O,GOOGL.O,JNJ.N,PFE.N
Date,,,,,,,,,,,,,,,,,
2019-01-02,3.40550,18.83,36.52,46.93,109.28,99.31,24.96,172.03,40.40,69.69,110.69,76.9565,101.12,135.68,52.7340,127.75,40.998794
2019-01-03,3.19975,17.05,34.36,46.64,108.26,97.11,24.56,169.51,39.68,68.62,108.57,75.0140,97.40,131.74,51.2735,125.72,39.851776
2019-01-04,3.40475,19.00,34.97,47.57,110.48,100.69,25.58,175.05,41.30,71.15,110.82,78.7695,101.93,137.95,53.9035,127.83,40.761807
2019-01-07,3.58500,20.57,35.23,46.95,109.53,100.76,25.56,176.02,41.71,71.52,112.26,81.4755,102.06,138.05,53.7960,127.01,40.979835
2019-01-08,3.49575,20.75,34.94,47.48,110.58,100.57,25.51,175.37,41.45,72.04,111.77,82.8290,102.80,142.53,54.2685,129.96,41.169425
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-12-22,48.83000,139.60,103.15,58.32,167.68,167.40,33.43,380.65,92.41,101.91,151.05,153.4200,374.58,353.39,141.4900,155.46,28.400000
2023-12-26,49.27900,143.41,104.45,58.56,168.86,168.39,33.86,381.61,92.84,102.14,152.41,153.4100,374.66,354.83,141.5200,156.14,28.410000
2023-12-27,49.41700,146.07,104.65,58.71,169.40,169.40,33.84,384.48,93.66,101.66,151.91,153.3400,374.07,357.83,140.3700,156.35,28.610000


In [4]:
display(cointegrated_pairs)

,y,x,hedge_ratio,intercept,adf_stat,p_value,is_cointegrated
0,CVX.N,XOM.N,0.713964,1.761593,-3.462651,0.009000,True
1,KO.N,PEP.O,0.664058,0.674101,-3.313506,0.014285,True
2,GS.N,MS.N,0.800362,2.256544,-3.252029,0.017164,True


## Build spread returns

In [22]:
spread_returns = build_spread_return_matrix(is_prices_df, cointegrated_pairs)

print(len(spread_returns))
print(spread_returns.shape)
print(spread_returns.describe())

1257
(1257, 3)
       CVX.N_vs_XOM.N  KO.N_vs_PEP.O  GS.N_vs_MS.N
count     1257.000000    1257.000000   1257.000000
mean         0.000118      -0.000022      0.000127
std          0.012361       0.009009      0.010138
min         -0.149677      -0.075451     -0.111712
25%         -0.004670      -0.004278     -0.004281
50%          0.000156       0.000064     -0.000042
75%          0.004652       0.004094      0.005180
max          0.136828       0.062758      0.065386


In [23]:
display(spread_returns)

,CVX.N_vs_XOM.N,KO.N_vs_PEP.O,GS.N_vs_MS.N
Date,,,
2019-01-03,-0.008191,0.000019,-0.000385
2019-01-04,-0.005600,0.006323,0.000006
2019-01-07,0.009281,-0.007323,-0.002404
2019-01-08,-0.009556,0.004923,0.001296
2019-01-09,0.009654,-0.000610,0.000673
...,...,...,...
2023-12-22,0.000993,0.002867,0.002111
2023-12-26,0.007392,-0.000558,-0.001202
2023-12-27,0.000075,0.000438,0.000452


Check for NaN

In [24]:
print(spread_returns.isnull().sum())

CVX.N_vs_XOM.N    0
KO.N_vs_PEP.O     0
GS.N_vs_MS.N      0
dtype: int64


## Build OU-implied expected returns

In [25]:
ou_mu = build_ou_expected_returns(is_prices_df, cointegrated_pairs, window=60, annualise=True, periods_per_year=252)

print(ou_mu)

CVX.N_vs_XOM.N   -0.039769
KO.N_vs_PEP.O     0.019327
GS.N_vs_MS.N     -0.106908
Name: ou_expected_return, dtype: float64


## Compute Historical Mean Baseline

In [26]:
tickers = list(dict.fromkeys(
    cointegrated_pairs['y'].tolist() + cointegrated_pairs['x'].tolist()
))
asset_returns = is_prices_df[tickers].pct_change().dropna()

hist_mu = historical_mean_return(asset_returns, annualise=True, periods_per_year=252)

print(hist_mu)

CVX.N    0.123651
KO.N     0.068765
GS.N     0.215655
XOM.N    0.131534
PEP.O    0.112049
MS.N     0.229549
dtype: float64


## Estimate covariance by computing covariance matrices

In [27]:
spread_cov = shrinkage_covariance(spread_returns, annualise=True)
asset_cov = shrinkage_covariance(asset_returns, annualise=True)

print(spread_cov)
print(asset_cov)

                CVX.N_vs_XOM.N  KO.N_vs_PEP.O  GS.N_vs_MS.N
CVX.N_vs_XOM.N        0.035263       0.003347      0.001371
KO.N_vs_PEP.O         0.003347       0.022899      0.000996
GS.N_vs_MS.N          0.001371       0.000996      0.026632
          CVX.N      KO.N      GS.N     XOM.N     PEP.O      MS.N
CVX.N  0.125685  0.032945  0.068893  0.101264  0.027776  0.077398
KO.N   0.032945  0.047113  0.031980  0.029732  0.033931  0.035518
GS.N   0.068893  0.031980  0.107261  0.061705  0.031383  0.098267
XOM.N  0.101264  0.029732  0.061705  0.117737  0.024232  0.068353
PEP.O  0.027776  0.033931  0.031383  0.024232  0.048305  0.035193
MS.N   0.077398  0.035518  0.098267  0.068353  0.035193  0.123593


## Sanity check - compare vectors side by side

In [28]:
comparison = pd.DataFrame({
    'ou_implied': ou_mu,
    'hist_mean': hist_mu
})
display(comparison)


,ou_implied,hist_mean
CVX.N,NaN,0.123651
CVX.N_vs_XOM.N,-0.039769,NaN
GS.N,NaN,0.215655
GS.N_vs_MS.N,-0.106908,NaN
KO.N,NaN,0.068765
KO.N_vs_PEP.O,0.019327,NaN
MS.N,NaN,0.229549
PEP.O,NaN,0.112049
XOM.N,NaN,0.131534


### Verify covariance matrices are positive definite

In [29]:
np.linalg.eigvalsh(spread_cov)  # all eigenvalues should be > 0

array([0.0219635 , 0.02646474, 0.03636637])

In [30]:
np.linalg.eigvalsh(asset_cov)

array([0.01350711, 0.01668187, 0.02016899, 0.05358185, 0.08284551,
       0.38290829])

### Check end of IN-SAMPLE z-score

In [31]:
for _, row in cointegrated_pairs.iterrows():
    spread = compute_spread(is_prices_df[row['y']], is_prices_df[row['x']], row['hedge_ratio'], row['intercept'])
    z = compute_zscore(spread, window=60).iloc[-1]
    print(f"{row['y']}_vs_{row['x']}: z = {z:.3f}, ou_mu = {ou_mu[f'{row[chr(121)]}_vs_{row[chr(120)]}']:.4f}")


CVX.N_vs_XOM.N: z = 0.297, ou_mu = -0.0398
KO.N_vs_PEP.O: z = -0.759, ou_mu = 0.0193
GS.N_vs_MS.N: z = 1.739, ou_mu = -0.1069


## Visaulise Table

In [32]:
validation_df = pd.DataFrame({
    'Pair': ['CVX/XOM', 'KO/PEP', 'GS/MS'],
    'End-of-IS Z-Score': [0.297, -0.759, 1.739],
    'OU-Implied Return (ann.)': [-0.0398, 0.0193, -0.1069],
    'Signal': ['Spread above mean', 'Spread below mean', 'Spread well above mean'],
    'Consistent': ['Yes', 'Yes', 'Yes']
})

display(validation_df)


,Pair,End-of-IS Z-Score,OU-Implied Return (ann.),Signal,Consistent
0,CVX/XOM,0.297,-0.0398,Spread above mean,Yes
1,KO/PEP,-0.759,0.0193,Spread below mean,Yes
2,GS/MS,1.739,-0.1069,Spread well above mean,Yes


In [33]:
print(validation_df.to_latex(index=False))

\begin{tabular}{lrrll}
\toprule
Pair & End-of-IS Z-Score & OU-Implied Return (ann.) & Signal & Consistent \\
\midrule
CVX/XOM & 0.297000 & -0.039800 & Spread above mean & Yes \\
KO/PEP & -0.759000 & 0.019300 & Spread below mean & Yes \\
GS/MS & 1.739000 & -0.106900 & Spread well above mean & Yes \\
\bottomrule
\end{tabular}



## Visualise Results

### Cumulative spread returns vs constituent assets

In [34]:
PAIR_META = [
    {"spread": "CVX.N_vs_XOM.N", "y": "CVX.N", "x": "XOM.N", "label": "CVX/XOM"},
    {"spread": "KO.N_vs_PEP.O",  "y": "KO.N",  "x": "PEP.O", "label": "KO/PEP"},
    {"spread": "GS.N_vs_MS.N",   "y": "GS.N",  "x": "MS.N",  "label": "GS/MS"},
]

fig1 = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    subplot_titles=[p["label"] for p in PAIR_META],
    vertical_spacing=0.08,
)

for i, pm in enumerate(PAIR_META, start=1):
    cum_spread = (1 + spread_returns[pm["spread"]]).cumprod()
    cum_y      = (1 + asset_returns[pm["y"]]).cumprod()
    cum_x      = (1 + asset_returns[pm["x"]]).cumprod()

    fig1.add_trace(go.Scatter(
        x=cum_spread.index, y=cum_spread.values,
        name=f"{pm['label']} Spread",
        line=dict(width=1.8), showlegend=(i == 1),
        legendgroup="spread",
    ), row=i, col=1)
    fig1.add_trace(go.Scatter(
        x=cum_y.index, y=cum_y.values,
        name=pm["y"],
        line=dict(width=1.2, dash="dot"), showlegend=True,
        legendgroup=pm["y"],
    ), row=i, col=1)
    fig1.add_trace(go.Scatter(
        x=cum_x.index, y=cum_x.values,
        name=pm["x"],
        line=dict(width=1.2, dash="dash"), showlegend=True,
        legendgroup=pm["x"],
    ), row=i, col=1)

    fig1.add_hline(
        y=1, line=dict(color="gray", dash="dash", width=0.8),
        annotation_text="Base" if i == 1 else None,
        annotation_font_size=9,
        annotation_position="right",
        row=i, col=1,
    )

fig1.update_yaxes(title_text="Cumulative Return", tickformat=".2f")
fig1.update_xaxes(title_text="Date", row=3, col=1)
fig1.update_layout(
    title="Cumulative Returns: Cointegrated Spread vs Constituent Assets (IS 2019\u20132023)",
    height=350 * 3,
    template="plotly_white",
    hovermode="x unified",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
        colorway=[
        "#1f77b4", "#d62728", "#2ca02c",   # spreads (blue, red, green)
        "#9467bd", "#8c564b", "#e377c2",   # y assets (purple, brown, pink)
        "#17becf", "#bcbd22", "#ff7f0e",   # x assets (cyan, olive, orange)
    ],
)
fig1.show()

#### Save to Matplotlib

In [35]:
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import scienceplots
plt.style.use(["science", "high-contrast"])
plt.rcParams["text.usetex"] = True

PAIRS = [
    {"spread": "CVX.N_vs_XOM.N", "y": "CVX.N", "x": "XOM.N", "label": "CVX/XOM"},
    {"spread": "KO.N_vs_PEP.O",  "y": "KO.N",  "x": "PEP.O", "label": "KO/PEP"},
    {"spread": "GS.N_vs_MS.N",   "y": "GS.N",  "x": "MS.N",  "label": "GS/MS"},
]

n_pairs = len(PAIRS)
panel_labels = [f"({chr(97 + i)})" for i in range(n_pairs)]
prop_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, axes = plt.subplots(nrows=n_pairs, ncols=1, figsize=(12, 4 * n_pairs), sharex=True)
if n_pairs == 1:
    axes = [axes]

for ax, pm, label, color in zip(axes, PAIRS, panel_labels, prop_cycle):
    first = (label == panel_labels[0])

    cum_s = (1 + spread_returns[pm["spread"]]).cumprod()
    cum_y = (1 + asset_returns[pm["y"]]).cumprod()
    cum_x = (1 + asset_returns[pm["x"]]).cumprod()

    ax.plot(cum_s.index, cum_s.values, linewidth=1.8, color=color,
            label=r"Spread" if first else None)
    ax.plot(cum_y.index, cum_y.values, linewidth=1.0, linestyle="--",
            label=pm["y"] if first else None)
    ax.plot(cum_x.index, cum_x.values, linewidth=1.0, linestyle=":",
            label=pm["x"] if first else None)
    ax.axhline(1, color="grey", linewidth=0.6, linestyle="--",
               label=r"Base" if first else None)

    ax.set_ylabel(r"Cum.\ Return")
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
    ax.set_title(label, loc="left", fontsize=10, fontweight="bold")

axes[-1].set_xlabel("Date")
axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="lower center", ncol=4,
           fontsize=9, bbox_to_anchor=(0.5, -0.02))
fig.tight_layout(rect=[0, 0.04, 1, 1])
fig.savefig("figures/03_cumulative_returns.pdf", bbox_inches="tight")
plt.close(fig)


### OU-implied vs historical $\mu$ bar chart

In [36]:

pair_labels  = ["CVX/XOM", "KO/PEP", "GS/MS"]
hist_tickers = hist_mu.index.tolist()

fig_mu = make_subplots(
    rows=1, cols=2,
    subplot_titles=["OU-Implied \u03bc (by Pair)", "Historical Mean \u03bc (by Ticker)"],
    horizontal_spacing=0.12,
)

fig_mu.add_trace(go.Bar(
    x=pair_labels,
    y=ou_mu.values,
    name="OU-Implied",
    showlegend=True,
), row=1, col=1)

fig_mu.add_trace(go.Bar(
    x=hist_tickers,
    y=hist_mu.values,
    name="Historical Mean",
    showlegend=True,
), row=1, col=2)

fig_mu.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=1, col=1)
fig_mu.add_hline(y=0, line=dict(color="black", width=0.8, dash="dash"), row=1, col=2)

fig_mu.update_yaxes(title_text="Expected Return (ann.)", tickformat=".1%", row=1, col=1)
fig_mu.update_yaxes(tickformat=".1%", row=1, col=2)
fig_mu.update_layout(
    title="Expected Return Estimates (Annualised, IS 2019\u20132023)",
    template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
)
fig_mu.show()

#### Save to Matplotlib

In [37]:
pair_labels  = ["CVX/XOM", "KO/PEP", "GS/MS"]
hist_tickers = hist_mu.index.tolist()
prop_cycle   = plt.rcParams["axes.prop_cycle"].by_key()["color"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.bar(pair_labels, ou_mu.values, color=prop_cycle[0], width=0.4)
ax1.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax1.set_title("(a)", loc="left", fontsize=10, fontweight="bold")
ax1.set_ylabel(r"Expected Return (ann.)")
ax1.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

ax2.bar(hist_tickers, hist_mu.values, color=prop_cycle[1], width=0.4)
ax2.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax2.set_title("(b)", loc="left", fontsize=10, fontweight="bold")
ax2.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=1))

fig.suptitle(r"Expected Return Estimates (Annualised, IS 2019--2023)", y=1.01)
fig.tight_layout()
fig.savefig("figures/03_mu_comparison.pdf", bbox_inches="tight")
plt.close(fig)


### Covariance/correlaton heatmap

In [45]:
import plotly.figure_factory as ff

corr = asset_cov / np.outer(np.sqrt(np.diag(asset_cov)), np.sqrt(np.diag(asset_cov)))
tickers_ordered = list(dict.fromkeys(
    [t for _, row in cointegrated_pairs.iterrows() for t in (row["y"], row["x"])]
))

corr_df = pd.DataFrame(corr, index=tickers_ordered, columns=tickers_ordered)

fig_corr = go.Figure(go.Heatmap(
    z=corr_df.values,
    x=corr_df.columns.tolist(),
    y=corr_df.index.tolist(),
    colorscale="RdBu_r",
    zmid=0, zmin=-1, zmax=1,
    text=np.round(corr_df.values, 2),
    texttemplate="%{text}",
    textfont=dict(size=11),
    colorbar=dict(title="Correlation"),
))
fig_corr.update_layout(
    title="Asset Return Correlation Matrix (IS 2019\u20132023, Shrinkage Covariance)",
    template="plotly_white",
    height=500,
    width=550,
    xaxis=dict(side="bottom"),
)
fig_corr.show()

#### Save to Matplotlib

In [46]:
corr = asset_cov / np.outer(np.sqrt(np.diag(asset_cov)), np.sqrt(np.diag(asset_cov)))
corr_df = pd.DataFrame(corr, index=tickers_ordered, columns=tickers_ordered)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr_df.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(tickers_ordered)))
ax.set_yticks(range(len(tickers_ordered)))
ax.set_xticklabels(tickers_ordered)
ax.set_yticklabels(tickers_ordered)

for i in range(len(tickers_ordered)):
    for j in range(len(tickers_ordered)):
        ax.text(j, i, f"{corr_df.values[i, j]:.2f}",
                ha="center", va="center", fontsize=8,
                color="white" if abs(corr_df.values[i, j]) > 0.6 else "black")

fig.colorbar(im, ax=ax, label="Correlation")
fig.tight_layout()
fig.savefig("figures/03_correlation_heatmap.pdf", bbox_inches="tight")
plt.close(fig)


## Save outputs

In [40]:
ou_mu.to_csv("../../data/processed/spread_mu.csv")
hist_mu.to_csv("../../data/processed//asset_mu.csv")
spread_cov.to_csv("../../data/processed/spread_cov.csv")
asset_cov.to_csv("../../data/processed/asset_cov.csv")
spread_returns.to_csv("../../data/processed/spread_returns.csv")
asset_returns.to_csv("../../data/processed/asset_returns.csv")
